# 02 · Modelo logístico completo e inferencia

Este cuaderno estima un modelo de regresión logística con los 13 predictores originales. Las variables `cp`, `restecg`, `slope` y `thal` se tratan como categóricas, evitando interpretar sus códigos como si fueran distancias lineales.

In [1]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")


In [2]:
from src.data import load_raw_data, clean_data
from src.modeling import fit_logit, coefficient_table, model_metrics, likelihood_ratio_block_tests
from src.config import PREDICTORS

clean, _ = clean_data(load_raw_data())
model = fit_logit(clean, PREDICTORS)
print(model.summary())

                           Logit Regression Results                           
Dep. Variable:                 target   No. Observations:                  302
Model:                          Logit   Df Residuals:                      282
Method:                           MLE   Df Model:                           19
Date:                Fri, 20 Nov 2026   Pseudo R-squ.:                  0.5179
Time:                        06:43:57   Log-Likelihood:                -100.38
converged:                       True   LL-Null:                       -208.21
Covariance Type:            nonrobust   LLR p-value:                 2.545e-35
                      coef    std err          z      P>|z|      [0.025      0.975]
-----------------------------------------------------------------------------------
Intercept           1.0067      3.354      0.300      0.764      -5.567       7.581
C(cp)[T.1]          0.9855      0.565      1.744      0.081      -0.122       2.093
C(cp)[T.2]          1.9090      

## Forma del modelo\n\nLa regresión logística modela el logaritmo de las *odds*:\n\n\[\log\left(\frac{p}{1-p}\right)=\beta_0+\beta_1x_1+\cdots+\beta_kx_k\]\n\nUn coeficiente positivo incrementa las log-odds de `target=1`; uno negativo las reduce, manteniendo los demás términos constantes. Para interpretación práctica se reportan también `exp(coef)`, es decir, los *odds ratios*.

In [3]:
coef = coefficient_table(model)
display(coef)

,term,coef,std_err,z,p_value,ci_95_low,ci_95_high,odds_ratio,or_ci_95_low,or_ci_95_high,significant_0_05
0,Intercept,1.0067,3.3541,0.3001,0.7641,-5.5672,7.5806,2.7365,0.0038,"1,959.7720",False
1,C(cp)[T.1],0.9855,0.5650,1.7442,0.0811,-0.1219,2.0929,2.6791,0.8852,8.1086,False
2,C(cp)[T.2],1.9090,0.4779,3.9946,0.0001,0.9724,2.8457,6.7464,2.6442,17.2129,True
3,C(cp)[T.3],2.0074,0.6498,3.0895,0.0020,0.7339,3.2809,7.4441,2.0833,26.5999,True
4,C(restecg)[T.1],0.5536,0.3744,1.4785,0.1393,-0.1803,1.2875,1.7395,0.8350,3.6236,False
5,C(restecg)[T.2],-0.3021,2.2828,-0.1324,0.8947,-4.7763,4.1720,0.7392,0.0084,64.8477,False
6,C(slope)[T.1],-0.7069,0.8579,-0.8239,0.4100,-2.3884,0.9747,0.4932,0.0918,2.6503,False
7,C(slope)[T.2],0.2312,0.9328,0.2478,0.8043,-1.5970,2.0594,1.2601,0.2025,7.8410,False
8,C(thal)[T.1],1.8291,2.3621,0.7744,0.4387,-2.8004,6.4587,6.2286,0.0608,638.2353,False
9,C(thal)[T.2],1.8568,2.2734,0.8168,0.4141,-2.5989,6.3125,6.4032,0.0744,551.4203,False


## Términos significativos al 5%

In [4]:
significant = coef[(coef["term"] != "Intercept") & (coef["p_value"] < 0.05)].copy()
significant = significant.sort_values("p_value")
display(significant[["term", "coef", "std_err", "z", "p_value", "ci_95_low", "ci_95_high", "odds_ratio"]])

,term,coef,std_err,z,p_value,ci_95_low,ci_95_high,odds_ratio
19,ca,-0.8817,0.2141,-4.1179,0.0000,-1.3013,-0.4620,0.4141
2,C(cp)[T.2],1.9090,0.4779,3.9946,0.0001,0.9724,2.8457,6.7464
3,C(cp)[T.3],2.0074,0.6498,3.0895,0.0020,0.7339,3.2809,7.4441
12,sex,-1.5188,0.5204,-2.9186,0.0035,-2.5387,-0.4989,0.2190
18,oldpeak,-0.4741,0.2259,-2.0983,0.0359,-0.9169,-0.0313,0.6225


## Métricas globales del ajuste

El pseudo-R² de McFadden no es un porcentaje de varianza explicada como el R² de OLS. Resume cuánto mejora la log-verosimilitud frente al modelo con solo intercepto.

In [5]:
metrics = model_metrics(model)
display(metrics)
vals = dict(zip(metrics["metric"], metrics["value"]))
print(f"Pseudo R² McFadden: {vals['pseudo_r2_mcfadden']:.4f}")
print(f"Log-Likelihood: {vals['log_likelihood']:.4f}")
print(f"LL-Null: {vals['ll_null']:.4f}")
print(f"LLR p-value: {vals['llr_p_value']:.3e}")

,metric,value
0,nobs,302.0000
1,df_model,19.0000
2,df_resid,282.0000
3,pseudo_r2_mcfadden,0.5179
4,log_likelihood,-100.3766
5,ll_null,-208.2099
6,llr,215.6665
7,llr_p_value,0.0000
8,aic,240.7532
9,bic,314.9617


Pseudo R² McFadden: 0.5179
Log-Likelihood: -100.3766
LL-Null: -208.2099
LLR p-value: 2.545e-35


### Interpretación

Un Log-Likelihood menos negativo que el LL-Null indica que los predictores mejoran el ajuste respecto al modelo nulo. El LLR p-value extremadamente pequeño aporta evidencia global contra la hipótesis de que todos los coeficientes de los predictores sean cero simultáneamente.

## Importancia por variable fuente

Las categorías generan varios coeficientes, por lo que comparar solamente un `z` individual puede ocultar el efecto conjunto de una variable. Como complemento se hacen pruebas de razón de verosimilitudes eliminando una variable fuente completa cada vez.

In [6]:
block_tests = likelihood_ratio_block_tests(clean, PREDICTORS)
display(block_tests)

,predictor,lr_stat,df,p_value,significant_0_05
11,ca,18.8642,1,0.0000,True
2,cp,21.2089,3,0.0001,True
1,sex,9.2256,1,0.0024,True
12,thal,12.7409,3,0.0052,True
9,oldpeak,4.6734,1,0.0306,True
8,exang,3.1940,1,0.0739,False
3,trestbps,2.8147,1,0.0934,False
10,slope,4.6610,2,0.0972,False
7,thalach,2.6218,1,0.1054,False
4,chol,1.1589,1,0.2817,False


In [7]:
terms_ranked = (coef[coef["term"] != "Intercept"]
                .assign(abs_z=lambda d: d["z"].abs())
                .sort_values("abs_z", ascending=False))
display(terms_ranked[["term", "z", "p_value", "coef", "odds_ratio"]].head(8))

,term,z,p_value,coef,odds_ratio
19,ca,-4.1179,0.0000,-0.8817,0.4141
2,C(cp)[T.2],3.9946,0.0001,1.9090,6.7464
3,C(cp)[T.3],3.0895,0.0020,2.0074,7.4441
12,sex,-2.9186,0.0035,-1.5188,0.2190
18,oldpeak,-2.0983,0.0359,-0.4741,0.6225
17,exang,-1.7979,0.0722,-0.7664,0.4647
1,C(cp)[T.1],1.7442,0.0811,0.9855,2.6791
13,trestbps,-1.6597,0.0970,-0.0178,0.9823


## Intervalo de confianza del término con mayor |z|

En este ajuste, `ca` presenta el mayor valor absoluto de `z` entre los términos individuales. Su intervalo se interpreta sobre la escala del coeficiente y, al exponenciarlo, sobre la escala de odds ratios.

In [8]:
ca_row = coef.loc[coef["term"] == "ca"].iloc[0]
print(f"coef(ca) = {ca_row['coef']:.4f}")
print(f"z = {ca_row['z']:.4f}; p = {ca_row['p_value']:.3e}")
print(f"IC95% coef = [{ca_row['ci_95_low']:.4f}, {ca_row['ci_95_high']:.4f}]")
print(f"OR = {ca_row['odds_ratio']:.4f}")
print(f"IC95% OR = [{ca_row['or_ci_95_low']:.4f}, {ca_row['or_ci_95_high']:.4f}]")

coef(ca) = -0.8817
z = -4.1179; p = 3.823e-05
IC95% coef = [-1.3013, -0.4620]
OR = 0.4141
IC95% OR = [0.2722, 0.6300]


## Lectura analítica

La evidencia estadística más consistente se concentra en `ca`, `cp`, `sex`, `thal` y `oldpeak` cuando se evalúan como variables fuente. Esto no implica causalidad: el estudio es observacional y el modelo describe asociaciones condicionadas a los predictores incluidos.